# Generador de reservas hoteleras sinteticas 2023-2026
### Hotel independiente en Baiona, Galicia — TFM RMS

Genera un dataset realista con:
- Tipo de habitacion **fijo** por numero de habitacion, sin solapes de fechas
- **Canal de reserva no uniforme**: Booking.com domina; walk-in/telefono suben en verano
- **Pais no uniforme**: España domina (+Portugal por cercania); mas turismo internacional en verano
- **Pickup mas rapido en verano**, mas lento en invierno
- Mas familias con niños en verano
- ADR por tipo de habitacion + estacionalidad + inflacion interanual

**Para usar en Google Colab**, ejecuta primero la celda de montaje de Drive y ajusta las rutas de guardado.

In [ ]:
# Si estas en Google Colab, descomenta estas lineas para montar Drive #

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

np.random.seed(42)

# Carpeta de salida (ajusta si usas Google Drive, ej. '/content/drive/MyDrive/TFM')
CARPETA_SALIDA = '/content/drive/MyDrive/Master/TFM/BBDD_Modelo'
CARPETA_SALIDA_MTD = '/content/drive/MyDrive/Master/TFM/Reservas_Hotel'
os.makedirs(CARPETA_SALIDA, exist_ok=True)
os.makedirs(CARPETA_SALIDA_MTD, exist_ok=True)

Mounted at /content/drive


## 1. Inventario de habitaciones
75 habitaciones, tipo **fijo** por numero (25 Individual + 25 Doble + 25 Suite).

In [ ]:
N_ROOMS = 75
TIPOS = ['Individual', 'Doble', 'Suite']
ROOM_TYPE = {}
for i, tipo in enumerate(TIPOS):
    for n in range(1, 26):
        ROOM_TYPE[i * 25 + n] = tipo

ADR_BASE_RANGE = {'Individual': (60, 100), 'Doble': (90, 140), 'Suite': (130, 200)}

## 2. Tendencia interanual
Volumen de reservas creciente + inflacion en ADR + estacionalidad mensual de demanda.

In [ ]:
N_RESERVAS = {2023: 4300, 2024: 4680, 2025: 5000, 2026: 5300}
ADR_FACTOR = {2023: 0.88, 2024: 0.94, 2025: 1.00, 2026: 1.05}

P_CANCELADA = 0.026
P_NOSHOW = 0.0227
P_CONFIRMADA = 1 - P_CANCELADA - P_NOSHOW

# Estacionalidad mensual de demanda (pesos relativos)
PESOS_MES = {1: 0.6, 2: 0.7, 3: 0.85, 4: 1.0, 5: 1.1, 6: 1.25, 7: 1.4, 8: 1.4, 9: 1.1, 10: 1.0, 11: 0.9, 12: 1.5}

VERANO = {6, 7, 8}
INVIERNO = {11, 12, 1, 2}

def estacion_de(mes):
    if mes in VERANO:
        return 'verano'
    if mes in INVIERNO:
        return 'invierno'
    return 'resto'

## 3. Canal de reserva
Distribucion base realista (Booking.com domina) + tendencia de digitalizacion interanual + estacionalidad
(mas walk-in/telefono en verano, mas OTA/agencia planificada en invierno).

In [ ]:
CANAL_BASE = {'Booking': 0.34, 'Web': 0.20, 'Expedia': 0.12,
              'Agencia': 0.09, 'Walk-in': 0.08, 'Teléfono': 0.17}

# Digitalizacion interanual: Web/Booking/Expedia ganan peso, Walk-in/Telefono/Agencia lo pierden
CANAL_SHIFT_ANIO = {
    2023: {'Web': 0.82, 'Booking': 0.92, 'Expedia': 0.85, 'Agencia': 1.05, 'Walk-in': 1.25, 'Teléfono': 1.20},
    2024: {'Web': 0.91, 'Booking': 0.96, 'Expedia': 0.93, 'Agencia': 1.02, 'Walk-in': 1.12, 'Teléfono': 1.10},
    2025: {c: 1.0 for c in CANAL_BASE},
    2026: {'Web': 1.08, 'Booking': 1.03, 'Expedia': 1.05, 'Agencia': 0.95, 'Walk-in': 0.88, 'Teléfono': 0.90},
}

# Estacionalidad: en verano sube el walk-in/telefono (turismo de paso por la costa), # en invierno domina la reserva planificada via OTA/agencia
CANAL_SHIFT_ESTACION = {
    'verano': {'Walk-in': 1.45, 'Teléfono': 1.25, 'Booking': 0.90, 'Web': 0.95, 'Agencia': 0.85, 'Expedia': 1.0},
    'invierno': {'Walk-in': 0.70, 'Teléfono': 0.85, 'Booking': 1.10, 'Web': 1.05, 'Agencia': 1.15, 'Expedia': 1.0},
    'resto': {c: 1.0 for c in CANAL_BASE},
}

def distribucion_canal(anio, mes):
    est = estacion_de(mes)
    ajustado = {c: p * CANAL_SHIFT_ANIO[anio].get(c, 1.0) * CANAL_SHIFT_ESTACION[est].get(c, 1.0)
                for c, p in CANAL_BASE.items()}
    total = sum(ajustado.values())
    return {c: p / total for c, p in ajustado.items()}

CANAL_DIST_CACHE = {(a, m): distribucion_canal(a, m) for a in N_RESERVAS for m in range(1, 13)}

## 4. Pais de origen
España domina (+Portugal por cercania fronteriza); el peso internacional
(Reino Unido, Alemania, Francia, Italia, EEUU) sube en verano.

In [ ]:
PAIS_BASE = {'España': 0.50, 'Portugal': 0.15, 'Francia': 0.09, 'Alemania': 0.08,
             'Reino Unido': 0.08, 'Italia': 0.06, 'EEUU': 0.04}

PAIS_SHIFT_ESTACION = {
    'verano': {'España': 0.90, 'Portugal': 0.95, 'Reino Unido': 1.35, 'Alemania': 1.35,
               'Francia': 1.25, 'Italia': 1.20, 'EEUU': 1.30},
    'invierno': {'España': 1.15, 'Portugal': 1.10, 'Reino Unido': 0.65, 'Alemania': 0.65,
                 'Francia': 0.75, 'Italia': 0.70, 'EEUU': 0.60},
    'resto': {c: 1.0 for c in PAIS_BASE},
}

def distribucion_pais(mes):
    est = estacion_de(mes)
    ajustado = {c: p * PAIS_SHIFT_ESTACION[est].get(c, 1.0) for c, p in PAIS_BASE.items()}
    total = sum(ajustado.values())
    return {c: p / total for c, p in ajustado.items()}

PAIS_DIST_CACHE = {m: distribucion_pais(m) for m in range(1, 13)}

## 5. Pickup (antelacion de reserva)
Mas rapido en verano (~28 dias de media), mas lento en invierno (~70 dias).

In [ ]:
LEAD_TIME_PARAMS = {
    'verano': (28, 18),     # (media, desviacion) en dias
    'invierno': (70, 30),
    'resto': (48, 25),
}

def lead_time_dias(mes, n=1):
    media, sd = LEAD_TIME_PARAMS[estacion_de(mes)]
    return np.clip(np.random.normal(media, sd, size=n).astype(int), 0, 150)

## 6. Niños y ADR
Mas familias con niños en verano. ADR depende del tipo de habitacion + estacionalidad + inflacion.

In [ ]:
def ninos_para(mes, n=1):
    if estacion_de(mes) == 'verano':
        return np.random.choice([0, 1, 2], size=n, p=[0.35, 0.35, 0.30])
    return np.random.choice([0, 1, 2], size=n, p=[0.65, 0.25, 0.10])

def adr_reserva(tipo, checkin_date, anio):
    lo, hi = ADR_BASE_RANGE[tipo]
    base = np.random.uniform(lo, hi)
    factor_mes = PESOS_MES[checkin_date.month]
    adr = base * ADR_FACTOR[anio] * (0.85 + 0.15 * factor_mes)
    return int(round(np.clip(adr, 45, 320)))

## 7. Calendario de ocupacion por habitacion
Genera, año a año y habitacion a habitacion, un calendario de estancias **sin solapes**
(las reservas Confirmada / No Show consumen inventario real; las gaps se reparten aleatoriamente).

In [ ]:
libre_desde = {r: pd.Timestamp('2023-01-01') for r in ROOM_TYPE}
registros_ocupados = []

for anio in [2023, 2024, 2025, 2026]:
    year_start = pd.Timestamp(f'{anio}-01-01')
    year_end = pd.Timestamp(f'{anio}-12-31')
    n_ocupadas_anio = int(round(N_RESERVAS[anio] * (P_CONFIRMADA + P_NOSHOW)))

    base, extra = divmod(n_ocupadas_anio, N_ROOMS)
    objetivo_por_room = {r: base for r in ROOM_TYPE}
    for r in np.random.choice(list(ROOM_TYPE), size=extra, replace=False):
        objetivo_por_room[r] += 1

    for room, k in objetivo_por_room.items():
        if k == 0:
            continue
        start = max(year_start, libre_desde[room])
        dias_disponibles = (year_end - start).days + 1
        if dias_disponibles <= 0:
            libre_desde[room] = start
            continue

        noches_list = np.random.randint(1, 8, size=k)
        while k > 0 and noches_list[:k].sum() >= dias_disponibles:
            k -= 1
        noches_list = noches_list[:k]
        if k == 0:
            libre_desde[room] = start
            continue

        hueco_total = dias_disponibles - noches_list.sum()
        cortes = np.sort(np.random.randint(0, hueco_total + 1, size=k))
        gaps = np.diff(np.concatenate(([0], cortes, [hueco_total])))

        cursor = start
        for i in range(k):
            cursor = cursor + pd.Timedelta(days=int(gaps[i]))
            checkin = cursor
            noches = int(noches_list[i])
            checkout = checkin + pd.Timedelta(days=noches)
            estado = np.random.choice(
                ['Confirmada', 'No Show'],
                p=[P_CONFIRMADA / (P_CONFIRMADA + P_NOSHOW), P_NOSHOW / (P_CONFIRMADA + P_NOSHOW)]
            )
            registros_ocupados.append({
                'anio': anio, 'Habitacion': room, 'Tipo': ROOM_TYPE[room],
                'CheckIn': checkin, 'CheckOut': checkout, 'Noches': noches, 'Estado': estado,
            })
            cursor = checkout
        libre_desde[room] = cursor

df_ocupadas = pd.DataFrame(registros_ocupados)
df_ocupadas.head()

,anio,Habitacion,Tipo,CheckIn,CheckOut,Noches,Estado
0,2023,1,Individual,2023-01-03,2023-01-08,5,Confirmada
1,2023,1,Individual,2023-01-10,2023-01-15,5,Confirmada
2,2023,1,Individual,2023-01-22,2023-01-24,2,No Show
3,2023,1,Individual,2023-01-25,2023-02-01,7,Confirmada
4,2023,1,Individual,2023-02-03,2023-02-08,5,Confirmada


## 8. Reservas Canceladas
No ocupan inventario real, asi que pueden coincidir en fechas con otra reserva (como en la vida real).

In [ ]:
def pesos_diarios_anio(anio):
    fechas = pd.date_range(f'{anio}-01-01', f'{anio}-12-31', freq='D')
    pesos = np.array([PESOS_MES[d.month] for d in fechas])
    return fechas, pesos / pesos.sum()

registros_cancel = []
for anio in [2023, 2024, 2025, 2026]:
    n_cancel = N_RESERVAS[anio] - int((df_ocupadas['anio'] == anio).sum())
    fechas, pesos = pesos_diarios_anio(anio)
    checkins = np.random.choice(fechas, size=n_cancel, p=pesos)
    for checkin in checkins:
        room = np.random.choice(list(ROOM_TYPE))
        noches = int(np.random.randint(1, 8))
        checkin = pd.Timestamp(checkin)
        registros_cancel.append({
            'anio': anio, 'Habitacion': room, 'Tipo': ROOM_TYPE[room],
            'CheckIn': checkin, 'CheckOut': checkin + pd.Timedelta(days=noches),
            'Noches': noches, 'Estado': 'Cancelada',
        })

df_cancel = pd.DataFrame(registros_cancel)
df_cancel.head()

,anio,Habitacion,Tipo,CheckIn,CheckOut,Noches,Estado
0,2023,25,Individual,2023-07-31,2023-08-04,4,Cancelada
1,2023,32,Doble,2023-05-18,2023-05-24,6,Cancelada
2,2023,55,Suite,2023-12-11,2023-12-15,4,Cancelada
3,2023,61,Suite,2023-12-29,2024-01-02,4,Cancelada
4,2023,57,Suite,2023-11-26,2023-12-01,5,Cancelada


## 9. Combinar todo
Añade Canal, Pais, ADR, antelacion de reserva y niños — todo dependiente del mes/estacion.

In [ ]:
df_all = pd.concat([df_ocupadas, df_cancel], ignore_index=True)
n = len(df_all)
meses = df_all['CheckIn'].dt.month.values

df_all['ADR'] = [adr_reserva(t, ci, a) for t, ci, a in zip(df_all['Tipo'], df_all['CheckIn'], df_all['anio'])]
df_all['Importe'] = df_all['ADR'] * df_all['Noches']

lead = np.array([lead_time_dias(m, 1)[0] for m in meses])
df_all['FechaReserva'] = (df_all['CheckIn'] - pd.to_timedelta(lead, unit='D')).dt.strftime('%Y-%m-%d')

df_all['Canal'] = [
    np.random.choice(list(CANAL_DIST_CACHE[(a, m)].keys()), p=list(CANAL_DIST_CACHE[(a, m)].values()))
    for a, m in zip(df_all['anio'], meses)
]
df_all['Pais'] = [
    np.random.choice(list(PAIS_DIST_CACHE[m].keys()), p=list(PAIS_DIST_CACHE[m].values()))
    for m in meses
]
df_all['Adultos'] = np.random.choice([1, 2], size=n, p=[0.5, 0.5])
df_all['Ninos'] = [ninos_para(m, 1)[0] for m in meses]
df_all['ClienteID'] = np.random.randint(1, 2001, size=n)

df_all['CheckIn'] = df_all['CheckIn'].dt.strftime('%Y-%m-%d')
df_all['CheckOut'] = df_all['CheckOut'].dt.strftime('%Y-%m-%d')

df_all = df_all.sort_values(['anio', 'CheckIn']).reset_index(drop=True)
df_all['ReservaID'] = range(1, n + 1)

COLUMNAS = ['ReservaID', 'ClienteID', 'FechaReserva', 'CheckIn', 'CheckOut', 'Noches',
            'Habitacion', 'Tipo', 'Canal', 'Pais', 'Adultos', 'Ninos', 'ADR', 'Importe', 'Estado']
df_final = df_all[COLUMNAS + ['anio']]
df_final.head()

,ReservaID,ClienteID,FechaReserva,CheckIn,CheckOut,Noches,Habitacion,Tipo,Canal,Pais,Adultos,Ninos,ADR,Importe,Estado,anio
0,1,377,2022-11-08,2023-01-01,2023-01-05,4,4,Individual,Web,Francia,2,1,58,232,Confirmada,2023
1,2,1549,2023-01-01,2023-01-01,2023-01-07,6,6,Individual,Web,Reino Unido,2,0,77,462,No Show,2023
2,3,530,2022-10-21,2023-01-01,2023-01-08,7,7,Individual,Booking,Italia,2,2,71,497,Confirmada,2023
3,4,1370,2022-11-24,2023-01-01,2023-01-08,7,11,Individual,Agencia,España,2,1,61,427,Confirmada,2023
4,5,1702,2022-09-23,2023-01-01,2023-01-04,3,16,Individual,Booking,España,2,1,76,228,Confirmada,2023


## 10. Verificaciones de integridad
Sin solapes de fechas por habitacion, y un unico tipo fijo por habitacion.

In [ ]:
def hay_solapes(df):
    ocupadas = df[df['Estado'].isin(['Confirmada', 'No Show'])].copy()
    ocupadas['CheckIn'] = pd.to_datetime(ocupadas['CheckIn'])
    ocupadas['CheckOut'] = pd.to_datetime(ocupadas['CheckOut'])
    for room, g in ocupadas.groupby('Habitacion'):
        g = g.sort_values('CheckIn')
        if (g['CheckIn'].shift(-1) < g['CheckOut']).any():
            return True
    return False

assert not hay_solapes(df_final), 'Se han detectado solapes'
assert df_final.groupby('Habitacion')['Tipo'].nunique().max() == 1, 'Una habitacion tiene mas de un tipo'
print('OK: sin solapes y tipo fijo por habitacion')

OK: sin solapes y tipo fijo por habitacion


## 11. Guardar
CSV combinado 2023-2025, y carpeta con 2026 troceado por mes.

In [ ]:
df_2023_2025 = df_final[df_final['anio'] < 2026].drop(columns='anio')
df_2023_2025.to_csv(f'{CARPETA_SALIDA}/BBDD_Hotel_Reservas_2023_2025_v3.csv', index=False)

DIR_2026 = f'{CARPETA_SALIDA}/2026_meses_v3'
os.makedirs(DIR_2026, exist_ok=True)
meses_es = ['01_Enero', '02_Febrero', '03_Marzo', '04_Abril', '05_Mayo', '06_Junio',
            '07_Julio', '08_Agosto', '09_Septiembre', '10_Octubre', '11_Noviembre', '12_Diciembre']
df_2026 = df_final[df_final['anio'] == 2026].drop(columns='anio').copy()
df_2026['_mes'] = pd.to_datetime(df_2026['CheckIn']).dt.month
for m in range(1, 13):
    sub = df_2026[df_2026['_mes'] == m].drop(columns='_mes')
    sub.to_csv(f'{DIR_2026}/BBDD_Hotel_Reservas_2026_{meses_es[m-1]}.csv', index=False)

print('Guardado en', CARPETA_SALIDA)

Guardado en /content/drive/MyDrive/Master/TFM/BBDD_Modelo


## 12. Resumen y comprobacion de los patrones de negocio

In [ ]:
print('--- Ocupacion y ADR por año ---')
for anio in [2023, 2024, 2025, 2026]:
    sub = df_final[df_final['anio'] == anio]
    noches_ocupadas = sub[sub['Estado'].isin(['Confirmada', 'No Show'])]['Noches'].sum()
    capacidad = N_ROOMS * (366 if anio == 2024 else 365)
    print(f'{anio}: {len(sub)} reservas | ADR medio {sub["ADR"].mean():.1f}€ | '
          f'ocupacion {noches_ocupadas/capacidad:.1%}')

--- Ocupacion y ADR por año ---
2023: 4300 reservas | ADR medio 107.1€ | ocupacion 61.5%
2024: 4680 reservas | ADR medio 114.0€ | ocupacion 66.9%
2025: 5000 reservas | ADR medio 121.7€ | ocupacion 70.9%
2026: 5300 reservas | ADR medio 127.5€ | ocupacion 75.6%


In [ ]:
print('--- Canal por estacion (2025) ---')
sub25 = df_final[df_final['anio'] == 2025].copy()
sub25['mes'] = pd.to_datetime(sub25['CheckIn']).dt.month
sub25['estacion'] = sub25['mes'].apply(estacion_de)
pd.crosstab(sub25['estacion'], sub25['Canal'], normalize='index').round(3)

--- Canal por estacion (2025) ---


Canal,Agencia,Booking,Expedia,Teléfono,Walk-in,Web
estacion,,,,,,
invierno,0.107,0.357,0.124,0.148,0.049,0.216
resto,0.095,0.332,0.131,0.170,0.073,0.198
verano,0.074,0.303,0.130,0.211,0.109,0.175


In [ ]:
print('--- Pais por estacion (2025) ---')
pd.crosstab(sub25['estacion'], sub25['Pais'], normalize='index').round(3)

--- Pais por estacion (2025) ---


Pais,Alemania,EEUU,España,Francia,Italia,Portugal,Reino Unido
estacion,,,,,,,
invierno,0.050,0.019,0.585,0.080,0.039,0.170,0.057
resto,0.073,0.035,0.514,0.094,0.057,0.142,0.085
verano,0.100,0.046,0.400,0.103,0.081,0.169,0.101


In [ ]:
print('--- Antelacion media (dias) por estacion (2025) ---')
sub25['lead'] = (pd.to_datetime(sub25['CheckIn']) - pd.to_datetime(sub25['FechaReserva'])).dt.days
sub25.groupby('estacion')['lead'].mean().round(1)

--- Antelacion media (dias) por estacion (2025) ---


,lead
estacion,
invierno,69.3
resto,46.6
verano,28.2


In [ ]:
print('--- % con niños por estacion (2025) ---')
(sub25.groupby('estacion')['Ninos'].apply(lambda s: (s > 0).mean())).round(3)

--- % con niños por estacion (2025) ---


,Ninos
estacion,
invierno,0.356
resto,0.371
verano,0.625


## 13. MTD (Month to Date) — snapshot de reservas "a fecha de hoy"
Filtra el año 2026 por **FechaReserva** (cuándo se hizo la reserva, no la estancia) hasta una fecha de corte.
Simula "cuántas reservas existen ya en el sistema" en un momento dado — útil para probar tu pipeline de n8n
con datos incrementales día a día.

Cambia `HOY` por la fecha que necesites y ejecuta la celda; puedes repetirla varias veces (ej. para generar
varios días seguidos, ver el bucle de la celda siguiente).

In [ ]:
# HOY = '2026-08-01'  # <-- cambia esta fecha por la que necesites

# df_2026_full = df_final[df_final['anio'] == 2026].drop(columns='anio').copy()
# df_2026_full['FechaReserva'] = pd.to_datetime(df_2026_full['FechaReserva'])

# hoy_ts = pd.Timestamp(HOY)
# mtd = df_2026_full[df_2026_full['FechaReserva'] <= hoy_ts].sort_values('FechaReserva').copy()
# mtd['FechaReserva'] = mtd['FechaReserva'].dt.strftime('%Y-%m-%d')

# nombre = f'BBDD_Hotel_Reservas_2026_{hoy_ts.strftime("%d%m%y")}.csv'
# ruta = f'{CARPETA_SALIDA}/{nombre}'
# mtd.to_csv(ruta, index=False)

# print(f'MTD a {HOY}: {len(mtd)} reservas -> {ruta}')

### 13b. (Opcional) Generar varios días de MTD de una vez
Útil para simular una semana entera de llegada incremental de reservas.

In [ ]:
FECHA_INICIO = '2026-10-01'
FECHA_FIN = '2026-10-25'

df_2026_full = df_final[df_final['anio'] == 2026].drop(columns='anio').copy()
df_2026_full['FechaReserva'] = pd.to_datetime(df_2026_full['FechaReserva'])

for fecha in pd.date_range(FECHA_INICIO, FECHA_FIN):
    mtd_dia = df_2026_full[df_2026_full['FechaReserva'] <= fecha].sort_values('FechaReserva').copy()
    mtd_dia['FechaReserva'] = mtd_dia['FechaReserva'].dt.strftime('%Y-%m-%d')
    nombre = f'BBDD_Hotel_Reservas_{fecha.strftime("%d%m%y")}.csv'
    mtd_dia.to_csv(f'{CARPETA_SALIDA_MTD}/{nombre}', index=False)
    print(f'{fecha.date()} ({fecha.day_name()}): {len(mtd_dia)} reservas -> {nombre}')

2026-10-01 (Thursday): 4930 reservas -> BBDD_Hotel_Reservas_011026.csv
2026-10-02 (Friday): 4942 reservas -> BBDD_Hotel_Reservas_021026.csv
2026-10-03 (Saturday): 4950 reservas -> BBDD_Hotel_Reservas_031026.csv
2026-10-04 (Sunday): 4957 reservas -> BBDD_Hotel_Reservas_041026.csv
2026-10-05 (Monday): 4970 reservas -> BBDD_Hotel_Reservas_051026.csv
2026-10-06 (Tuesday): 4983 reservas -> BBDD_Hotel_Reservas_061026.csv
2026-10-07 (Wednesday): 4991 reservas -> BBDD_Hotel_Reservas_071026.csv
2026-10-08 (Thursday): 5005 reservas -> BBDD_Hotel_Reservas_081026.csv
2026-10-09 (Friday): 5017 reservas -> BBDD_Hotel_Reservas_091026.csv
2026-10-10 (Saturday): 5032 reservas -> BBDD_Hotel_Reservas_101026.csv
2026-10-11 (Sunday): 5045 reservas -> BBDD_Hotel_Reservas_111026.csv
2026-10-12 (Monday): 5061 reservas -> BBDD_Hotel_Reservas_121026.csv
2026-10-13 (Tuesday): 5069 reservas -> BBDD_Hotel_Reservas_131026.csv
2026-10-14 (Wednesday): 5079 reservas -> BBDD_Hotel_Reservas_141026.csv
2026-10-15 (Thursd